# Implementing the RAG Flow (`003_vectordb.ipynb`)

En este notebook implementamos el flujo completo de **Retrieval-Augmented Generation (RAG)** en 5 pasos:
1. **Chunking:** Dividir el texto por secciones (`chunk_by_section`).
2. **Embeddings:** Generar representaciones vectoriales para cada chunk.
3. **Vector Store:** Almacenar los embeddings junto con el texto original.
4. **Query Embedding:** Generar el embedding de la pregunta del usuario.
5. **Similarity Search:** Buscar los chunks más relevantes mediante distancia coseno.

## Setup y Funciones Auxiliares
Configuración del cliente, función de *chunking* por secciones, generación de embeddings (con soporte para *batches*) e implementación de `VectorIndex`.

**Sin tarjeta en Voyage:** deja `USE_LOCAL_EMBEDDINGS = True` (gratis, corre en tu Mac). Para el curso con API: `False` + `VOYAGE_API_KEY`.

Si usas local: intenta `sentence-transformers`; si falla Hugging Face (`HfFolder`), el notebook usa **lite** automáticamente.

Opcional (arreglar Hugging Face en el mismo Python del kernel): `import sys; !{sys.executable} -m pip install -U huggingface_hub sentence-transformers`

In [76]:
# Client Setup
import os
import re
import math
from anthropic import Anthropic

# True = embeddings en tu máquina (sin Voyage ni rate limits). False = Voyage del curso.
USE_LOCAL_EMBEDDINGS = True

client = Anthropic()
vo = None
if not USE_LOCAL_EMBEDDINGS:
    import voyageai
    vo = voyageai.Client()  # Requiere VOYAGE_API_KEY
    print("Embeddings: Voyage API")
else:
    print("Embeddings: local (sentence-transformers o fallback lite sin Hugging Face)")

Embeddings: local (sentence-transformers o fallback lite sin Hugging Face)


In [77]:
# Chunk by section
def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [78]:
# Embedding Generation (soporta string individual o lista de strings)
import hashlib

LOCAL_EMBED_DIM = 384
_local_backend = None  # "sentence_transformers" | "lite"
_local_st_model = None


def _lite_embed(text: str, dim: int = LOCAL_EMBED_DIM) -> list:
    """Embedding simple sin dependencias (práctica RAG si Hugging Face falla)."""
    vec = [0.0] * dim
    for token in re.findall(r"\w+", text.lower()):
        bucket = int(hashlib.md5(token.encode()).hexdigest(), 16) % dim
        vec[bucket] += 1.0
    mag = math.sqrt(sum(x * x for x in vec))
    if mag == 0:
        return vec
    return [x / mag for x in vec]


def _local_embed_batch(texts):
    global _local_backend, _local_st_model
    if _local_backend is None:
        try:
            from sentence_transformers import SentenceTransformer

            _local_st_model = SentenceTransformer("all-MiniLM-L6-v2")
            _local_backend = "sentence_transformers"
            print("Local backend: sentence-transformers (all-MiniLM-L6-v2)")
        except Exception as e:
            _local_backend = "lite"
            print(
                "Local backend: lite (sin Hugging Face).",
                f"{type(e).__name__}: {str(e)[:100]}",
            )

    if _local_backend == "sentence_transformers":
        return (
            _local_st_model.encode(texts, normalize_embeddings=True).tolist()
        )
    return [_lite_embed(t) for t in texts]


def generate_embedding(chunks, model="voyage-3-large", input_type=None):
    is_list = isinstance(chunks, list)
    input_data = chunks if is_list else [chunks]

    if USE_LOCAL_EMBEDDINGS:
        vecs = _local_embed_batch(input_data)
        return vecs if is_list else vecs[0]

    result = vo.embed(input_data, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [79]:
# VectorIndex implementation
class VectorIndex:
    def __init__(self, distance_metric="cosine", embedding_fn=None):
        self.vectors = []
        self.data = []
        self.distance_metric = distance_metric
        self.embedding_fn = embedding_fn

    def add_vector(self, vector, data):
        self.vectors.append(vector)
        self.data.append(data)

    def add_document(self, document):
        if not self.embedding_fn:
            raise ValueError("Embedding function not provided during initialization.")
        if not isinstance(document, dict) or "content" not in document:
            raise ValueError("Document must be a dictionary with a 'content' key.")
        vector = self.embedding_fn(document["content"])
        self.add_vector(vector, document)

    def search(self, query, k=1):
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self.embedding_fn:
                raise ValueError("Embedding function not provided for string query.")
            query_vector = self.embedding_fn(query)
        elif isinstance(query, list) and all(isinstance(x, (int, float)) for x in query):
            query_vector = query
        else:
            raise TypeError("Query must be either a string or a list of numbers.")

        if len(query_vector) != len(self.vectors[0]):
            raise ValueError(
                f"Query vector dimension ({len(query_vector)}) must match stored vectors ({len(self.vectors[0])})"
            )

        if self.distance_metric == "cosine":
            dist_func = self._cosine_distance
        elif self.distance_metric == "euclidean":
            dist_func = self._euclidean_distance
        else:
            raise ValueError(f"Unsupported distance metric: {self.distance_metric}")

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.data[i]))

        distances.sort(key=lambda item: item[0])
        return [(data, distance) for distance, data in distances[:k]]

    def _euclidean_distance(self, vec1, vec2):
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1, vec2):
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec):
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1, vec2):
        dot_prod = self._dot_product(vec1, vec2)
        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)
        if mag1 == 0 or mag2 == 0:
            return 1.0
        cosine_similarity = max(-1.0, min(1.0, dot_prod / (mag1 * mag2)))
        return 1.0 - cosine_similarity

    def __len__(self):
        return len(self.vectors)

## Step 1: Chunking the Text By Section

Cargamos `./report.md` y lo dividimos en secciones lógicas.

In [80]:
with open("./report_vd.md", "r", encoding="utf-8") as f:
    text = f.read()

chunks = chunk_by_section(text)

print(f"{len(chunks)} secciones indexables:")
for i, c in enumerate(chunks, 1):
    title = c.splitlines()[0][:70] if c.strip() else "(vacío)"
    print(f"  {i}. {title}")

12 secciones indexables:
  1. # Annual Interdisciplinary Research Review: Cross-Domain Insights
  2. Table of Contents
  3. Section 1: Executive Summary & Cross-Domain Overview
  4. Section 2: Software Engineering - Project Phoenix Stability Enhancemen
  5. Section 3: Financial Analysis - Q4 Performance and Outlook
  6. Section 4: Cybersecurity - Incident Response and Threat Mitigation
  7. Section 5: Human Resources - Talent Acquisition and Retention Metrics
  8. Section 6: Product Engineering - Roadmap Execution and Dependencies
  9. Section 7: Data Science & Analytics - Predictive Modeling Initiatives
  10. Section 8: Legal & Compliance - Regulatory Audit Findings
  11. Section 9: Operations & Supply Chain - Logistics Optimization
  12. Section 10: Customer Success - Support Ticket Resolution Trends


## Step 2: Generate Embeddings
Generamos los embeddings para todos los chunks en una sola llamada (*batch*).

Usar `input_type="document"` (Voyage) mejora la búsqueda vs mezclar tipos.

**Orden:** siempre Step 1 → 2 → 3. Si cambias `chunks`, vuelve a correr 2 y 3.

In [81]:
embeddings = generate_embedding(chunks, input_type="document")
print(f"{len(embeddings)} embeddings · dim={len(embeddings[0])}")

Error importing huggingface_hub.hf_api: cannot import name 'HfFolder' from 'huggingface_hub.utils' (/Users/pablo.gomez/Library/Python/3.9/lib/python/site-packages/huggingface_hub/utils/__init__.py)
Local backend: lite (sin Hugging Face). ImportError: cannot import name 'HfFolder' from 'huggingface_hub.utils' (/Users/pablo.gomez/Library/Python/3.9/li
12 embeddings · dim=384


## Step 3: Store in Vector Database
Creamos el `VectorIndex` y guardamos cada embedding acompañado del texto original (`{"content": chunk}`).

Ejecutar **después** de Step 2 en la misma sesión (mismo `embeddings` y `chunks`).

In [82]:
assert len(embeddings) == len(chunks), "Re-ejecuta Step 2 (embeddings) antes de indexar."

store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})

print(f"Índice listo: {len(store)} chunks · dim={len(store.vectors[0])}")

Índice listo: 12 chunks · dim=384


## Step 4: Process User Queries
Convertimos la pregunta del usuario en un vector de embedding.

In [83]:
QUERY = (
    "What platform engineering team has done over the past four quarters?"
)
user_embedding = generate_embedding(QUERY, input_type="query")

## Step 5: Find Relevant Content
Buscamos los 2 chunks más cercanos en el espacio vectorial (menor distancia coseno = mayor similitud).

In [84]:
results = store.search(user_embedding, k=2)

for rank, (doc, distance) in enumerate(results, 1):
    title = doc["content"].splitlines()[0][:60]
    print(f"#{rank}  distance={distance:.4f}  ·  {title}")
    print(doc["content"][:400], "\n" + "-" * 60)

#1  distance=0.6584  ·  Section 2: Software Engineering - Project Phoenix Stability 
Section 2: Software Engineering - Project Phoenix Stability Enhancements
The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems underpinning Project Phoenix. Recurring issues, particularly `ERR_MEM_ALLOC_FAIL_0x8007000E` during peak loads and `TIMEOUT_QUERY_DB_0xDEADBEEF` affecting data retrieval operations, were prioritized,  
------------------------------------------------------------
#2  distance=0.7091  ·  Section 6: Product Engineering - Roadmap Execution and Depen
Section 6: Product Engineering - Roadmap Execution and Dependencies
Product Engineering delivered three major enterprise workflow enhancements while managing critical upstream dependencies with the Project Phoenix core storage layer (detailed in Section 2). Cross-team integration testing uncovered latency bottlenecks under concurrent batch exports, prompting joint 

In [85]:
QUERY = (
    "What happened with INC-2023-Q4-011?"
)
user_embedding = generate_embedding(QUERY, input_type="query")

In [86]:
results = store.search(user_embedding, k=2)

for rank, (doc, distance) in enumerate(results, 1):
    title = doc["content"].splitlines()[0][:60]
    print(f"#{rank}  distance={distance:.4f}  ·  {title}")
    print(doc["content"][:400], "\n" + "-" * 60)

#1  distance=0.7145  ·  Section 2: Software Engineering - Project Phoenix Stability 
Section 2: Software Engineering - Project Phoenix Stability Enhancements
The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems underpinning Project Phoenix. Recurring issues, particularly `ERR_MEM_ALLOC_FAIL_0x8007000E` during peak loads and `TIMEOUT_QUERY_DB_0xDEADBEEF` affecting data retrieval operations, were prioritized,  
------------------------------------------------------------
#2  distance=0.9055  ·  Section 8: Legal & Compliance - Regulatory Audit Findings
Section 8: Legal & Compliance - Regulatory Audit Findings
Annual internal and third-party compliance audits concluded with zero critical findings across data privacy and financial reporting controls. Updated vendor risk assessments and data retention policies were ratified to align with upcoming international regulatory frameworks taking effect in mid-2025.
 
--------

# legal no tiene que ver por eso lexical search es nedcesario.